# Union Mask and Binary-Mutation Clustering Analysis
Workflow completo per: caricamento dati, costruzione di `X_bin`, clustering nello spazio originale, visualizzazioni PCA e analisi di qualita dei cluster.

In [ ]:
import adabmDCA
import copy
import time
import os
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# ===============================================================================
# Import adabmDCA utility functions
# ===============================================================================
# These functions provide core functionality for:
# - FASTA file I/O and sequence handling
# - Statistical analysis (frequencies, correlations)
# - One-hot encoding of sequences
# - Parameter and chain initialization
# - MCMC sampling
# - Energy computations
# ===============================================================================

from adabmDCA.fasta import get_tokens, import_from_fasta, compute_weights
from adabmDCA.stats import get_freq_single_point, get_freq_two_points
from adabmDCA.functional import one_hot
from adabmDCA.utils import init_parameters, init_chains, get_device, get_dtype
from adabmDCA.sampling import get_sampler

from adabmDCA.io import load_params, load_chains
# from adabmDCA.resampling import compute_mixing_time, compute_seqID, get_mean_seqid
from adabmDCA.utils import resample_sequences
from adabmDCA.statmech import compute_energy

# ===============================================================================
# Configure computational device and data type
# ===============================================================================
device = get_device("cuda")      # Use GPU for accelerated computation
dtype = get_dtype("float32")     # Use 32-bit floating point precision


## 1) Data Loading
Caricamento dei dataset `DN`, `DT-`, `DT+` e definizione dei token RNA.

In [ ]:
tokens = get_tokens("rna")  # Standard protein alphabet (20 amino acids + gap)
headers_DN, data_DN = import_from_fasta(
    "Group_I_intron-2/DN&DT/DN.fasta", 
    tokens=tokens, 
    filter_sequences=True  # Remove problematic sequences
)
headers_DTm, data_DTm = import_from_fasta(
    "Group_I_intron-2/DN&DT/DT-.fasta", 
    tokens=tokens, 
    filter_sequences=True  # Remove problematic sequences
)
headers_DTp, data_DTp = import_from_fasta(
    "Group_I_intron-2/DN&DT/DT+.fasta", 
    tokens=tokens, 
    filter_sequences=True  # Remove problematic sequences
)
# output_path = "../images/"

## 3) Subsampling and Initial Setup
Preparazione del campione `DT_subsample` per le analisi successive.

In [ ]:
rna_vector = np.array([
    0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 
    1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 
    0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 
    0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 
    1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 
    1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 
    0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 
    1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 
    0, 1, 1, 0, 0
])

rna_vector = rna_vector[3:-1]
print("Lunghezza array:", len(rna_vector))


plt.bar(range(len(rna_vector)), 1- rna_vector)
plt.show()

In [ ]:
DT = np.concatenate((data_DTm, data_DTp), axis=0)

# Use the full DT dataset (no subsampling), preserving order: first DTm then DTp.
DT_full = np.concatenate((data_DTm, data_DTp), axis=0)
n_dtm = data_DTm.shape[0]
n_dtp = data_DTp.shape[0]



rng = np.random.default_rng()
idx = rng.choice(DT.shape[0], size=100, replace=False)
DT_subsample = DT[idx, :]
print(f"DT_subsample shape: {DT_subsample.shape}")

In [ ]:
print(f"Red: {n_dtm / DT.shape[0]:.3f}, Green: {n_dtp / DT.shape[0]:.3f}")

In [ ]:
DT_full

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

plot_dir = Path("nucleotide_space_plots")
plot_dir.mkdir(exist_ok=True)

q = 5
DT_full_int = np.asarray(DT_full, dtype=np.int64)
if DT_full_int.min() < 0 or DT_full_int.max() >= q:
    raise ValueError(f"DT_full contains symbols outside the expected range [0, {q-1}].")

DT_full_onehot = np.eye(q, dtype=np.uint8)[DT_full_int].reshape(DT_full_int.shape[0], -1)

# Keep track of sequence provenance in DT_full.
is_dtm_full = np.zeros(DT_full_int.shape[0], dtype=bool)
is_dtm_full[:data_DTm.shape[0]] = True
is_dtp_full = ~is_dtm_full
idx_dtm_full = np.flatnonzero(is_dtm_full)
idx_dtp_full = np.flatnonzero(is_dtp_full)
dt_group_labels = np.where(is_dtm_full, "DTm", "DTp")

kmeans9 = KMeans(n_clusters=9, random_state=0, n_init=20)
dt_kmeans9_labels = kmeans9.fit_predict(DT_full_onehot)

pca_dt = PCA(n_components=2, random_state=0)
DT_full_pca = pca_dt.fit_transform(DT_full_onehot)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    DT_full_pca[:, 0],
    DT_full_pca[:, 1],
    c=dt_kmeans9_labels,
    cmap="tab10",
    s=16,
    alpha=0.8,
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("DT_full one-hot: KMeans with 9 clusters")
kmeans_pca_path = plot_dir / "dt_full_kmeans9_pca.png"
fig.savefig(kmeans_pca_path, dpi=300, bbox_inches="tight")
print(f"Saved {kmeans_pca_path}")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    DT_full_pca[is_dtp_full, 0],
    DT_full_pca[is_dtp_full, 1],
    color="green",
    s=16,
    alpha=0.7,
    label="data_DTp",
)
ax.scatter(
    DT_full_pca[is_dtm_full, 0],
    DT_full_pca[is_dtm_full, 1],
    color="red",
    s=16,
    alpha=0.7,
    label="data_DTm",
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PC1 vs PC2 colored by DTm/DTp membership")
ax.legend()
membership_pca_path = plot_dir / "dt_full_pca_dtm_dtp.png"
fig.savefig(membership_pca_path, dpi=300, bbox_inches="tight")
print(f"Saved {membership_pca_path}")
plt.show()

In [ ]:
reference_DN0 = np.asarray(data_DN[0], dtype=np.int64)
if DT_full_int.shape[1] != reference_DN0.shape[0]:
    raise ValueError(
        f"Length mismatch: DT_full has L={DT_full_int.shape[1]}, reference_DN0 has L={reference_DN0.shape[0]}"
    )

# Binary encoding relative to the first DN sequence: 1 if the site matches DN[0], 0 otherwise.
DT_vs_DN0_bin = (DT_full_int != reference_DN0).astype(np.uint8)

cluster_ids = np.arange(kmeans9.n_clusters)
cluster_cmap = plt.get_cmap("tab10", kmeans9.n_clusters)
cluster_match_freqs = []

fig, axes = plt.subplots(3, 3, figsize=(20, 12), sharex=True, sharey=True)
axes = axes.ravel()

for cluster_id, ax in zip(cluster_ids, axes):
    cluster_mask = dt_kmeans9_labels == cluster_id
    cluster_bin = DT_vs_DN0_bin[cluster_mask]
    dtp_frac = is_dtp_full[cluster_mask].mean() if np.any(cluster_mask) else np.nan

    if cluster_bin.shape[0] == 0:
        freq_ones = np.full(DT_vs_DN0_bin.shape[1], np.nan)
        ax.set_title(f"Cluster {cluster_id} (n=0, DTp={100 * dtp_frac:.1f}%)")
        ax.axis("off")
        cluster_match_freqs.append(freq_ones)
        continue

    freq_ones = cluster_bin.mean(axis=0)
    cluster_match_freqs.append(freq_ones)

    ax.bar(
        np.arange(freq_ones.shape[0]),
        freq_ones,
        color=cluster_cmap(cluster_id),
        width=1.0,
    )
    ax.set_title(f"Cluster {cluster_id} (n={cluster_bin.shape[0]}, DTp={100 * dtp_frac:.1f}%)")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Site")
    ax.set_ylabel("Freq. of 1")

cluster_match_freqs = np.asarray(cluster_match_freqs)
fig.suptitle("Frequency of 1 per site in each cluster (1 = mismatch with data_DN[0])")
plt.tight_layout()
cluster_hist_path = plot_dir / "dt_clusters_site_freq_histograms.png"
fig.savefig(cluster_hist_path, dpi=300, bbox_inches="tight")
print(f"Saved {cluster_hist_path}")
plt.show()